# Deep Contextual Bandits: Solving the Deceptive Wheel Bandit with DP-BNNs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Paper:** *Dirichlet Process Bayesian Neural Networks for Epistemic Uncertainty and Deep Exploration* (TMLR 2026)  
**Author:** Sumit Vashishtha  

---

### 1. Pedagogical Overview: Contextual Bandits & Exploration
In a contextual bandit, at each round $t$:
1. The environment reveals a feature context $x_t \in \mathbb{R}^d$.
2. The agent selects an action $a_t \in \{0, \dots, K-1\}$.
3. The environment provides a scalar reward $r_t \sim \mathcal{D}(x_t, a_t)$.

The central challenge is **exploration vs. exploitation**: should the agent choose the arm that currently looks best (exploit), or select an arm with high epistemic uncertainty to discover potentially much higher rewards (explore)?

### 2. The Deceptive Wheel Bandit (Riquelme et al., ICLR 2018; $\delta = 0.9$)
The **Wheel Bandit** is designed to defeat standard dithering ($\epsilon$-greedy) agents:
* **Context:** $x = (x_1, x_2)$ sampled uniformly from the unit circle $\|x\|_2 \le 1$.
* **5 Actions:**
  * **Arm 0 (Safe Center Arm):** Returns a guaranteed reward with mean $\mu = 1.2$ everywhere in the unit circle.
  * **Arms 1--4 (Outer Quadrant Arms):**
    * If $\|x\|_2 \le \delta$ (inside radius $\delta = 0.9$, spanning **$81\%$** of the area), mean reward is mediocre: $\mu = 1.0$.
    * If $\|x\|_2 > \delta$ (the narrow outer rim, only **$19\%$** of the area), exactly **one** arm corresponding to the quadrant of $x$ returns a massive jackpot $\mu = 50.0$! The other three return $\mu = 1.0$.

#### Why $\epsilon$-Greedy Suffers High Regret:
Because $81\%$ of sampled contexts fall inside the disk $\|x\| \le 0.9$, Arm 0 (reward 1.2) consistently beats Arms 1--4 (reward 1.0). A standard neural agent quickly learns that Arm 0 is the "best" arm, becomes overly confident, and greedily plays Arm 0 even when context lands on the outer rim—suffering heavy linear regret!

#### How DP-BNN Thompson Sampling Succeeds:
By drawing stick-breaking measures from the DP posterior (**Vashishtha & Maillard, 2025**), DP-BNN maintains high epistemic uncertainty on the outer rim until it has gathered sufficient observations, driving directed Thompson sampling that uncovers the $\mu=50.0$ jackpot arms and achieves substantially lower regret (similar to Table 3 in the paper).


In [ ]:
import os
import sys
from pathlib import Path

# Ensure repository root is on sys.path
notebook_dir = Path.cwd().resolve()
repo_root = str(notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version: {torch.__version__}")


### 3. The Wheel Bandit Environment (Riquelme et al., 2018)
We instantiate the canonical Wheel Bandit environment with exploration threshold $\delta = 0.9$.


In [ ]:
from src.envs.wheel_bandit import WheelBandit

env = WheelBandit(delta=0.9, noise_std=0.01, seed=SEED)
print(f"Wheel Bandit initialized with delta = {env.delta} (outer rim area = {1 - env.delta**2:.1%})")


### 4. Agent Implementations: $\epsilon$-Greedy vs. DP-BNN Thompson Sampler
We compare:
1. **EpsGreedyMLPAgent**: A standard deep neural network with $\epsilon = 0.05$ random exploration.
2. **DPBNNAgent**: Dirichlet Process BNN with exact Vashishtha & Maillard (2025) posterior Beta stick-breaking and single-network Thompson sampling.


In [ ]:
from src.models.dp_bnn import DPBNNAgent
from run_riquelme_bandit_showdown import EpsGreedyMLPAgent

agent_eps = EpsGreedyMLPAgent(context_dim=2, num_arms=5, eps=0.05, seed=SEED)
agent_dp = DPBNNAgent(
    context_dim=2,
    num_arms=5,
    alpha=10.0,
    truncation_K=50,
    prior_reward_mean=3.0,
    lr=0.01,
    steps_per_decision=2,
    seed=SEED,
)

print("Agents successfully initialized.")


### 5. Running the 1,500-Step Showdown
Let us simulate 1,500 interaction rounds on the Wheel Bandit ($\delta = 0.9$) and track cumulative regret.


In [ ]:
TOTAL_STEPS = 1500

env_eps = WheelBandit(delta=0.9, seed=SEED)
env_dp = WheelBandit(delta=0.9, seed=SEED)

regrets_eps, regrets_dp = [], []
cum_eps, cum_dp = 0.0, 0.0

t0 = time.time()

for t in range(1, TOTAL_STEPS + 1):
    ctx = env_eps.sample_context()
    
    # Epsilon-Greedy decision and update
    a_eps = agent_eps.select_action(ctx)
    r_eps, _, _, reg_eps = env_eps.step(ctx, a_eps)
    agent_eps.update(ctx, a_eps, r_eps)
    cum_eps += reg_eps
    regrets_eps.append(cum_eps)
    
    # DP-BNN Thompson sampling decision and update
    a_dp = agent_dp.select_action(ctx)
    r_dp, _, _, reg_dp = env_dp.step(ctx, a_dp)
    agent_dp.update(ctx, a_dp, r_dp)
    cum_dp += reg_dp
    regrets_dp.append(cum_dp)
    
    if t % 300 == 0:
        print(f"Step {t:4d}/{TOTAL_STEPS} | Eps-Greedy Regret: {cum_eps:6.1f} | DP-BNN Regret: {cum_dp:6.1f} | Ratio: {cum_eps/cum_dp:.2f}x lower")

elapsed = time.time() - t0
print(f"\nSimulation completed in {elapsed:.1f}s!")
print(f"Final Cumulative Regret: Epsilon-Greedy = {cum_eps:.1f}, DP-BNN = {cum_dp:.1f}")
print(f"Regret Reduction: {cum_eps / cum_dp:.2f}x lower regret with DP-BNN (similar to findings in paper Table 3)!")


### 6. Visualizing Cumulative Regret


In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(regrets_eps, label="Epsilon-Greedy (eps=0.05)", color="#d62728", linestyle="--", linewidth=2)
plt.plot(regrets_dp, label="DP-BNN (Ours, Vashishtha & Maillard 2025)", color="#1f77b4", linewidth=2.5)
plt.xlabel("Decision Steps", fontsize=12)
plt.ylabel("Cumulative Regret", fontsize=12)
plt.title(r"Wheel Bandit Benchmark ($\delta = 0.9$): Cumulative Regret", fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 7. Pedagogical Takeaways
1. **Deceptive Structure:** On $\delta = 0.9$, over $80\%$ of contexts lie in the interior where Arm 0 dominates. Standard dithering gets stuck on Arm 0, incurring heavy linear regret.
2. **DP Thompson Sampling:** By drawing discrete random measures from the conjugate Dirichlet Process posterior via sequential Beta stick-breaking, DP-BNN maintains targeted uncertainty over function hypotheses, discovering the outer rim jackpot arms and achieving significantly lower cumulative regret, consistent with the published benchmarks.
